In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [50]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [42]:
index

False

In [51]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"content": 2.0, "filename": 0.5},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [20]:
from pathlib import Path
import sys

root = Path.cwd()
if (root / 'rag_helper_hw.py').exists():
    sys.path.insert(0, str(root))
else:
    sys.path.insert(0, str(root / 'lessons' / 'lesson01' / 'homework'))

from rag_helper_hw import RAGBase
from openai import OpenAI

In [52]:
openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)


In [53]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

('The loop keeps calling the model in a `while True` block, and after each call it checks whether the model returned any `function_call` items.\n\n- If there is a function call, the code runs the tool, appends the tool result to `messages`, and continues the loop.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls in the model’s response**.', ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7208))


In [41]:
answer


('It keeps calling the model in a `while True` loop.\n\nAfter each model response, the code checks whether the response contains any `function_call` items:\n\n- If it does, it runs the tool, appends the tool output to `messages`, and loops again.\n- If it does not, it breaks out of the loop.\n\nSo the stop condition is:\n\n- **no function calls in the latest response** → done\n\nIn the lesson code, that’s the `has_function_calls` flag:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the agent loop keeps going until the model returns a final message without asking for any more tools.',
 ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=141, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7252))

In [23]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [24]:
len(chunks)

295

In [54]:
index_chunk = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunk.fit(chunks)

In [55]:
assistant_chunk = RAGBase(
    index=index_chunk,
    llm_client=openai_client,
)

In [62]:
answer = assistant_chunk.rag("where is the search_tool function defined?")
print(answer)

('The `search_tool` is defined in `01-agentic-rag/lessons/13-function-calling.md` as a Python dictionary named `search_tool`.', ResponseUsage(input_tokens=2456, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=37, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2493))


In [58]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [63]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [73]:
def search(query, num_results=5):
        

        return index.search(
            query,
            num_results=num_results,
        )

In [74]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [75]:
INSTRUCTIONS = '''You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [76]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [77]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
